# 0. Extract of CBS Kerncijfers wijken en buurten 2024

Three of our variables come from CBS *Kerncijfers wijken en buurten 2024* (table 85984NED): education level, population density and the share of apartments. The full CBS download is very large, because it contains every district and neighbourhood in the Netherlands. We only need the 342 municipalities and five variables.

In this notebook we make a small table with **one row per municipality**, which the other notebooks use.

**Input** (folder `data/raw/`):
- `kwb2024.zip`: the full CBS download (CSV), with the files `Observations.csv` (all values) and `WijkenEnBuurtenCodes.csv` (the names that belong to the codes)

**Output** (folder `data/raw/`):
- `kwb2024_per_gemeente.csv`: one row per municipality with the CBS municipality code, the name and the five variables

The extract is saved next to the other input files in `data/raw/`, so notebooks 2.2, 5 and 6 can read it from there. Run this notebook before those notebooks.

In [1]:
import zipfile
import pandas as pd

RAW = "../data/raw/"

## Step 1. Open the CBS download

The CBS download is a zip file with several CSV files in it. With `zipfile` we can read a file from the zip without unpacking it first. We first print which files are in the zip.

In [2]:
zip_file = zipfile.ZipFile(RAW + "kwb2024.zip")

for name in zip_file.namelist():
    print(name)

Catalog.csv
D-85984NED.json
Dimensions.csv
Distribution.csv
LinkProvenances.csv
LinkRelatedSources.csv
MeasureCodes.csv
MeasureGroups.csv
Observations.csv
Properties.csv
WijkenEnBuurtenCodes.csv
WijkenEnBuurtenGroups.csv


We need two of these files. Depending on how the zip was made, they can sit in a subfolder, so we look for the file name at the end of each path.

In [3]:
for name in zip_file.namelist():
    if name.endswith("Observations.csv"):
        observations_file = name
    if name.endswith("WijkenEnBuurtenCodes.csv"):
        codes_file = name

print("Values:", observations_file)
print("Names: ", codes_file)

Values: Observations.csv
Names:  WijkenEnBuurtenCodes.csv


## Step 2. Load all values

`Observations.csv` is a long table: every row is one value for one area and one variable. The most important columns are:
- `WijkenEnBuurten`: the code of the area. Municipalities start with `GM`, districts with `WK` and neighbourhoods with `BU`;
- `Measure`: the code of the variable;
- `Value`: the value.

We read everything as text (`dtype=str`), so that codes such as `GM0014` keep their leading zeros. We turn the values into numbers later.

In [4]:
obs = pd.read_csv(zip_file.open(observations_file), sep=";", dtype=str)

print("Number of rows:", len(obs))
obs.head()

Number of rows: 1959396


,Id,Measure,WijkenEnBuurten,Value,StringValue,ValueAttribute
0,0,GM000C,NL00,NaN,Nederland,NaN
1,1,R00003,NL00,NaN,Land,NaN
2,2,R00004,NL00,NaN,NL00,NaN
3,3,T001036,NL00,17942942,NaN,NaN
4,4,3000,NL00,8920768,NaN,NaN


## Step 3. Keep only municipalities and the five variables

The CBS codes of the variables we need are below. We use this dictionary twice: to select the rows, and to give the columns readable names.

In [5]:
KWB_MEASURES = {
    "M000100": "bevolkingsdichtheid",   # inhabitants per km² of land
    "ZW10340": "meergezinswoning_pct",  # apartments as % of all dwellings
    "2018700": "opl_laag",              # persons 15-75 with a low education level
    "2018740": "opl_middel",            # persons 15-75 with a middle education level
    "2018790": "opl_hoog",              # persons 15-75 with a high education level (hbo, wo)
}

# Only municipalities (code starts with GM)
obs = obs[obs["WijkenEnBuurten"].str.startswith("GM")]

# Only the five variables we need
obs = obs[obs["Measure"].isin(KWB_MEASURES)]

print("Number of rows left:", len(obs))

Number of rows left: 1709


## Step 4. One row per municipality

Now every municipality still has five rows, one per variable. With `pivot` we turn this into a table with one row per municipality and one column per variable. Then we replace the CBS codes in the column names with our own names, and turn the text into numbers. Values that CBS does not publish become `NaN` (missing).

In [6]:
extract = obs.pivot(index="WijkenEnBuurten", columns="Measure", values="Value")
extract = extract.rename(columns=KWB_MEASURES)
extract = extract.apply(pd.to_numeric, errors="coerce")

extract.head()

Measure,opl_laag,opl_middel,opl_hoog,bevolkingsdichtheid,meergezinswoning_pct
WijkenEnBuurten,,,,,
GM0014,31740.0,78560,89290,1314,61
GM0034,47430.0,74710,54260,1753,30
GM0037,7480.0,11820,4090,272,21
GM0047,6330.0,9920,4280,362,21
GM0050,4200.0,8570,5890,97,18


## Step 5. Add the municipality names

The table only has codes such as `GM0014`. The names are in `WijkenEnBuurtenCodes.csv`: the column `Identifier` has the code and the column `Title` has the name. We use `loc` to look up the name for each code, and put it in a new column at the front.

In [7]:
codes = pd.read_csv(zip_file.open(codes_file), sep=";", dtype=str)
codes = codes.set_index("Identifier")

extract.insert(0, "gemeente", codes.loc[extract.index, "Title"].str.strip().values)
extract.index.name = "gemeentecode"
extract = extract.reset_index()

print("Number of municipalities:", len(extract))
extract.head()

Number of municipalities: 342


Measure,gemeentecode,gemeente,opl_laag,opl_middel,opl_hoog,bevolkingsdichtheid,meergezinswoning_pct
0,GM0014,Groningen,31740.0,78560,89290,1314,61
1,GM0034,Almere,47430.0,74710,54260,1753,30
2,GM0037,Stadskanaal,7480.0,11820,4090,272,21
3,GM0047,Veendam,6330.0,9920,4280,362,21
4,GM0050,Zeewolde,4200.0,8570,5890,97,18


## Step 6. Check and save

We check that there are 342 municipalities and see which values are missing. Then we save the table.

In [8]:
missing = extract.isna().sum()
print(missing[missing > 0])

extract[extract.isna().any(axis=1)]

Measure
opl_laag    1
dtype: int64


Measure,gemeentecode,gemeente,opl_laag,opl_middel,opl_hoog,bevolkingsdichtheid,meergezinswoning_pct
70,GM0277,Rozendaal,NaN,480,690,66,2


In [9]:
extract.to_csv(RAW + "kwb2024_per_gemeente.csv", index=False)
print("Saved:", RAW + "kwb2024_per_gemeente.csv")

Saved: ../data/raw/kwb2024_per_gemeente.csv
